In [107]:
import pandas as pd

In [108]:
df = pd.read_csv("/Users/laurent/Documents/Charles/DALAS-project/data/processed/civ_targets.csv", delimiter=";")
df.head()

,Country,Admin1,Admin2,ISO3,Admin2 Pcode,Admin1 Pcode,Month,Year,Events,Fatalities
0,Ukraine,Luhansk,Luhanskyi,UKR,UA4406,UA44,January,2018,1,0
1,Ukraine,Zakarpattia,Tiachivskyi,UKR,UA2108,UA21,January,2018,0,0
2,Ukraine,Kherson,Skadovskyi,UKR,UA6508,UA65,January,2018,0,0
3,Ukraine,Donetsk,Horlivskyi,UKR,UA1406,UA14,January,2018,0,0
4,Ukraine,Crimea,Yevpatoriiskyi,UKR,UA0108,UA01,January,2018,0,0


In [109]:
df = df.drop(columns=["Country", "ISO3"])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14595 entries, 0 to 14594
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Admin1        14595 non-null  str  
 1   Admin2        14385 non-null  str  
 2   Admin2 Pcode  14595 non-null  str  
 3   Admin1 Pcode  14595 non-null  str  
 4   Month         14595 non-null  str  
 5   Year          14595 non-null  int64
 6   Events        14595 non-null  int64
 7   Fatalities    14595 non-null  int64
dtypes: int64(3), str(5)
memory usage: 912.3 KB


In [110]:
df = df.drop(
    df[df["Admin1"].isin(["Black Sea", "Sea of Azov", "Crimea"])].index
)
df["Admin1"] = df["Admin1"].replace({"C": "Cherkasy"})

sorted(df["Admin1"].unique())

['Cherkasy',
 'Chernihiv',
 'Chernivtsi',
 'Dnipropetrovsk',
 'Donetsk',
 'Ivano-Frankivsk',
 'Kharkiv',
 'Kherson',
 'Khmelnytskyi',
 'Kirovograd',
 'Kyiv',
 'Kyiv City',
 'Luhansk',
 'Lviv',
 'Mykolaiv',
 'Odesa',
 'Poltava',
 'Rivne',
 'Sumy',
 'Ternopil',
 'Vinnytsia',
 'Volyn',
 'Zakarpattia',
 'Zaporizhia',
 'Zhytomyr']

In [111]:
df["Admin2"].unique()

<StringArray>
[            'Luhanskyi',           'Tiachivskyi',            'Skadovskyi',
            'Horlivskyi',             'Romenskyi',            'Fastivskyi',
            'Poltavskyi',             'Lubenskyi',   'Volodymyr-Volynskyi',
               'Sumskyi',
 ...
          'Rovenkivskyi',    'Novohrad-Volynskyi',           'Nikopolskyi',
        'Bohodukhivskyi',          'Beryslavskyi',              'Umanskyi',
 'Bilhorod-Dnistrovskyi',     'Mohyliv-Podilskyi',          'Kremenetskyi',
   'Kamianets-Podilskyi']
Length: 127, dtype: str

In [112]:
df_edu = pd.read_csv("/Users/laurent/Documents/Charles/DALAS-project/data/processed/education_core_wide.csv")
sorted(df_edu["Region"].unique())

['Cherkaska',
 'Chernihivska',
 'Chernivetska',
 'Dnipropetrovska',
 'Donetska',
 'Ivano-Frankivska',
 'Kharkivska',
 'Khersonska',
 'Khmelnytska',
 'Kirovohradska',
 'Kyiv',
 'Kyivska',
 'Luhanska',
 'Lvivska',
 'Mykolaivska',
 'Odeska',
 'Poltavska',
 'Rivnenska',
 'Sumska',
 'Ternopilska',
 'Vinnytska',
 'Volynska',
 'Zakarpatska',
 'Zaporizka',
 'Zhytomyrska']

In [113]:
df_mapping = pd.DataFrame({
    "Region": sorted(df_edu["Region"].unique()),
    "Admin1": sorted(df["Admin1"].unique())
})

# Le tri alphabétique inverse Kyiv (ville) et Kyivska (oblast) : on corrige par swap
i_ville = df_mapping.index[df_mapping["Region"] == "Kyiv"][0]
i_oblast = df_mapping.index[df_mapping["Region"] == "Kyivska"][0]

a = df_mapping.at[i_ville, "Admin1"]
b = df_mapping.at[i_oblast, "Admin1"]
df_mapping.at[i_ville, "Admin1"], df_mapping.at[i_oblast, "Admin1"] = b, a

print(df_mapping)

              Region           Admin1
0          Cherkaska         Cherkasy
1       Chernihivska        Chernihiv
2       Chernivetska       Chernivtsi
3    Dnipropetrovska   Dnipropetrovsk
4           Donetska          Donetsk
5   Ivano-Frankivska  Ivano-Frankivsk
6         Kharkivska          Kharkiv
7         Khersonska          Kherson
8        Khmelnytska     Khmelnytskyi
9      Kirovohradska       Kirovograd
10              Kyiv        Kyiv City
11           Kyivska             Kyiv
12          Luhanska          Luhansk
13           Lvivska             Lviv
14       Mykolaivska         Mykolaiv
15            Odeska            Odesa
16         Poltavska          Poltava
17         Rivnenska            Rivne
18            Sumska             Sumy
19       Ternopilska         Ternopil
20         Vinnytska        Vinnytsia
21          Volynska            Volyn
22       Zakarpatska      Zakarpattia
23         Zaporizka       Zaporizhia
24       Zhytomyrska         Zhytomyr


In [114]:
df_join = pd.DataFrame(
    columns=['Region', 'Year', 'preschool_children', 'preschool_employees',
       'preschool_institutions', 'preschool_places', 'pretertiary_enrolled',
       'pretertiary_graduated', 'pretertiary_institutions',
       'pretertiary_students', 'secondary_institutions', 'secondary_pupils',
       'secondary_teachers', 'tertiary_enrolled', 'tertiary_graduated',
       'tertiary_institutions', 'tertiary_students', 'vocational_enrolled',
       'vocational_graduated', 'vocational_institutions', 'vocational_pupils', 'Events', 'Fatalities']
)

# Pour la jointure, il faut faire la chose suivante :
# Pour civ target, il faut calculer pour chaque région, pour chaque année donnée, la somme des events et des fatalities
# Sachant que les mois ne sont pas pris en considération dans la dataset education_core_wide, on est obligé de faire ainsi, et on perd donc de l'information
# Par ailleurs, on ne pourra pas avoir le niveau administratif de niveau 2 dans le dataframe résultant de la jointure, 
# car dans le dataset sur l'education, le niveau de granularité ne s'arrête qu'aux régions
# On doit aller de Region vers Region, puis on récupère l'Admin1 pour pouvoir ensuite récupérer la somme des Events et des Fatalities


df_groupby = df.groupby(["Admin1", "Year"])[["Events", "Fatalities"]].sum().reset_index()
df_mapping.loc[
    df_mapping["Admin1"] == df_groupby["Admin1"].iloc[0],
    "Region"
]

# df_edu.iloc[0]["Region"]
for i in range(df_edu.shape[0]):
    admin1 = df_mapping.loc[
                df_mapping["Region"] == df_edu.iloc[i]["Region"],
                "Admin1"
            ].values[0]

    year = df_edu.iloc[i]["Year"]

    result = df_groupby.loc[
        (df_groupby["Admin1"] == admin1) &
        (df_groupby["Year"] == year),
        ["Events", "Fatalities"]
    ]

    events, fatalities = result[["Events", "Fatalities"]].values[0]

    df_join.loc[i] = {
        **df_edu.iloc[i].to_dict(),
        "Events": events,
        "Fatalities": fatalities
    }

df_join.info()

<class 'pandas.DataFrame'>
Index: 175 entries, 0 to 174
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Region                    175 non-null    str    
 1   Year                      175 non-null    int64  
 2   preschool_children        173 non-null    float64
 3   preschool_employees       173 non-null    float64
 4   preschool_institutions    173 non-null    float64
 5   preschool_places          173 non-null    float64
 6   pretertiary_enrolled      139 non-null    float64
 7   pretertiary_graduated     139 non-null    float64
 8   pretertiary_institutions  138 non-null    float64
 9   pretertiary_students      139 non-null    float64
 10  secondary_institutions    175 non-null    float64
 11  secondary_pupils          175 non-null    float64
 12  secondary_teachers        175 non-null    float64
 13  tertiary_enrolled         163 non-null    float64
 14  tertiary_graduated        

In [115]:
df_join.to_csv("main.csv", index=False)
df_join.describe()

,Year,preschool_children,preschool_employees,preschool_institutions,preschool_places,pretertiary_enrolled,pretertiary_graduated,pretertiary_institutions,pretertiary_students,secondary_institutions,...,tertiary_enrolled,tertiary_graduated,tertiary_institutions,tertiary_students,vocational_enrolled,vocational_graduated,vocational_institutions,vocational_pupils,Events,Fatalities
count,175.000000,173.000000,173.000000,173.000000,173.000000,139.000000,139.000000,138.000000,139.000000,175.000000,...,163.000000,163.000000,163.000000,163.000000,175.000000,175.000000,175.000000,175.000000,175.000000,175.000000
mean,2022.000000,39119.768786,11328.410405,563.410405,42309.930636,4680.510791,3226.589928,11.536232,13417.834532,536.262857,...,16533.092025,12869.245399,17.134969,47229.588957,4761.360000,4450.040000,27.171429,9281.440000,98.691429,72.262857
std,2.005739,21130.153479,5530.414542,198.795060,21151.773768,3078.244087,1888.043435,6.252777,8407.085605,202.769587,...,18652.109522,15025.335463,17.956572,55048.129557,2335.303436,2073.352037,11.092831,4279.904117,268.827567,220.004832
min,2019.000000,208.000000,181.000000,16.000000,274.000000,384.000000,375.000000,1.000000,1273.000000,77.000000,...,3335.000000,2036.000000,3.000000,10181.000000,2.000000,487.000000,11.000000,988.000000,0.000000,0.000000
25%,2020.000000,26495.000000,8037.000000,449.000000,28417.000000,2666.500000,2052.500000,7.000000,8260.000000,409.500000,...,7020.500000,5253.500000,7.500000,19698.000000,3322.500000,3088.500000,20.000000,6719.500000,1.000000,0.000000
50%,2022.000000,33464.000000,9869.000000,555.000000,36367.000000,3876.000000,2685.000000,11.000000,11210.000000,541.000000,...,9549.000000,7988.000000,11.000000,27109.000000,4550.000000,4317.000000,24.000000,8799.000000,5.000000,1.000000
75%,2024.000000,46227.000000,14079.000000,711.000000,53399.000000,5472.000000,3737.000000,14.000000,15510.500000,629.500000,...,16160.000000,12643.000000,18.000000,42478.000000,5516.000000,5170.500000,32.000000,10893.500000,35.000000,30.500000
max,2025.000000,107547.000000,27999.000000,1003.000000,101816.000000,16304.000000,10369.000000,33.000000,41201.000000,1194.000000,...,106334.000000,90103.000000,97.000000,341282.000000,13456.000000,12536.000000,58.000000,21896.000000,1441.000000,2079.000000
